# Adaptive Gauss-Kronrod check (Python)

Checks `thole.quad.gauss_kronrod` against mpmath and `scipy.integrate.quad`, and shows how the subdivision goes.
Start JupyterLab with `workshop run -- notebook`. After changing the Rust code, restart it.

In [ ]:
import math
import mpmath
from scipy import integrate
import matplotlib.pyplot as plt
from thole.quad import gauss_kronrod, ConvergenceError, Integral

ATOL = RTOL = 1.49e-8   # the defaults, those of scipy.integrate.quad

## 1. A first call

In [ ]:
r = gauss_kronrod(math.exp, 0, 1)
print(r)
print("error:", abs(r.value - (math.e - 1)))
print("hex (compare with the Rust notebook):", r.value.hex())

## 2. Against mpmath (50 digits) and SciPy

SciPy's `quad` uses QUADPACK's `qags`: a 21-point rule, extrapolation and heuristics for rounding, so it needs fewer evaluations on singular integrands. Both are within the tolerance.

In [ ]:
mpmath.mp.dps = 50
problems = [
    ("exp(x)",              math.exp,                              0, 1, mpmath.e - 1),
    ("sqrt(x)",             math.sqrt,                             0, 1, mpmath.mpf(2) / 3),
    ("-log(x)",             lambda x: -math.log(x),                0, 1, 1),
    ("|x - 1/3|",           lambda x: abs(x - 1 / 3),             -1, 1, mpmath.mpf(10) / 9),
    ("1/(1e-4+(x-.3)^2)",   lambda x: 1 / (1e-4 + (x - 0.3) ** 2), 0, 1, 100 * (mpmath.atan(70) + mpmath.atan(30))),
    ("cos(30x)",            lambda x: math.cos(30 * x),            0, 1, mpmath.sin(30) / 30),
]
print(f"{'f':20}{'thole':>22}{'error':>10}{'estimate':>10}{'evals':>7}{'scipy error':>13}{'evals':>7}")
for name, f, a, b, exact in problems:
    r = gauss_kronrod(f, a, b)
    s, _, info = integrate.quad(f, a, b, full_output=True)
    print(f"{name:20}{r.value:22.16f}{float(abs(r.value - exact)):10.1e}{r.error:10.1e}{r.evaluations:7d}{float(abs(s - exact)):13.1e}{info['neval']:7d}")

## 3. What the definitions and theorems predict

`gk_partition` and `gk_length`: the subintervals partition `[a, b]`, and there are `(evaluations / 15 + 1) / 2` of them. `gk_error_le_of_local`: when every local error estimate bounds the local error, so does their sum. The estimate is not a bound in general; it is for these integrands.

In [ ]:
f, exact = math.sqrt, 2 / 3
for t in [1e-3, 1e-6, 1e-9, 1e-12]:
    r = gauss_kronrod(f, 0, 1, atol=t, rtol=0, max_subintervals=1000)
    print(f"tol={t:<6g} value={r.value:.16f} error={abs(r.value - exact):.1e} estimate={r.error:.1e} subintervals={r.subintervals}")
    assert r.evaluations == 15 * (2 * r.subintervals - 1)
    assert abs(r.value - exact) <= r.error <= t

## 4. Where the points go

Record every evaluation: the subdivision concentrates them at the singularity of `sqrt` at 0 and around the peak at 0.3.

In [ ]:
for name, f in [("sqrt(x)", math.sqrt), ("peak at 0.3", lambda x: 1 / (1e-4 + (x - 0.3) ** 2))]:
    xs = []
    gauss_kronrod(lambda x: (xs.append(x), f(x))[1], 0, 1, atol=1e-12, rtol=0, max_subintervals=1000)
    plt.hist(xs, bins=200, alpha=0.6, label=f"{name}: {len(xs)} evaluations")
plt.xlabel("x"); plt.ylabel("evaluations"); plt.legend(); plt.show()

## 5. An estimate, not a bound

A peak narrower than the spacing of the 15 nodes is invisible to both rules: they agree, the error estimate is zero, and the value is wrong.

In [ ]:
r = gauss_kronrod(lambda x: 1 + (1e6 if abs(x - 0.55) < 1e-6 else 0), 0, 1)
print(r, "  true value: about 3")

## 6. Failures

In [ ]:
for name, call in {
    "infinite end":       lambda: gauss_kronrod(math.exp, 0, math.inf),
    "negative tolerance": lambda: gauss_kronrod(math.exp, 0, 1, atol=-1),
    "zero limit":         lambda: gauss_kronrod(math.exp, 0, 1, max_subintervals=0),
    "f returns inf":      lambda: gauss_kronrod(lambda x: 1 / x if x else math.inf, -1, 1),
    "limit too small":    lambda: gauss_kronrod(math.sqrt, 0, 1, atol=1e-14, rtol=0, max_subintervals=3),
    "f raises":           lambda: gauss_kronrod(lambda x: 1 / 0, 0, 1),
}.items():
    try:
        print(f"{name:20} -> returned {call()!r}")
    except Exception as e:
        print(f"{name:20} -> {type(e).__name__}: {e}")